# 00 — Environment and Data Check

This notebook is the entry point for the full workflow. It checks the local compiler, MPI, OpenMP build, correctness tests, and the four binary CSR datasets. Later notebooks use the same verified `build/pagerank` executable and `.prcsr` data.

The multi-gigabyte graph files are not included in the submission. On a new computer, run `bash scripts/setup_environment.sh` and `bash scripts/prepare_data.sh all` from the project root before executing this notebook. Saved outputs remain available for inspection without downloading the data.

Run order: **environment → build and tests → dataset sizes → file integrity**.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / 'Makefile').exists():
    ROOT = ROOT.parent
assert (ROOT / 'Makefile').exists(), 'Start Jupyter from the project directory or its notebook subdirectory'
os.chdir(ROOT)
print('Project root located; working directory set successfully.')

import hashlib
import json
import platform
import shutil
import subprocess
import pandas as pd

def command_output(*args):
    executable = shutil.which(args[0])
    if executable is None:
        return 'NOT FOUND'
    result = subprocess.run([executable, *args[1:]], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return result.stdout.strip().splitlines()[0] if result.stdout.strip() else executable


Project root located; working directory set successfully.


## 1. Software Environment

The project requires C++17, MPI, and OpenMP. macOS uses Homebrew Open MPI and `libomp`; Vanda uses the `openmpi/gcc/64/4.1.8` module.

In [2]:
environment = pd.Series({
    'OS': platform.platform(),
    'machine': platform.machine(),
    'Python': platform.python_version(),
    'C++ compiler': command_output('c++', '--version'),
    'MPI compiler': command_output('mpicxx', '--version'),
    'MPI runtime': command_output('mpirun', '--version'),
    'make': command_output('make', '--version'),
})
display(environment.to_frame('value'))
assert shutil.which('c++') and shutil.which('mpicxx') and shutil.which('mpirun')


,value
OS,macOS-26.6.2-arm64-arm-64bit-Mach-O
machine,arm64
Python,3.14.0
C++ compiler,Apple clang version 21.0.0 (clang-2100.4.1.1)
MPI compiler,Apple clang version 21.0.0 (clang-2100.4.1.1)
MPI runtime,mpirun (Open MPI) 5.0.11
make,GNU Make 3.81


## 2. Build and Correctness Tests

All three modes share the CSR loader, dangling-node handling, stopping rule, and metrics format.

| Mode | Configuration | Parallel mechanism |
|---|---|---|
| `serial` | `1×1` | One process |
| `mpi` | `R×1` | MPI ranks |
| `hybrid` | `R×T` | MPI ranks × OpenMP threads/rank |

The next cell rebuilds the executable and tests serial, MPI-only, MPI+OpenMP, isolated vertices, and cases with more ranks than vertices.

In [3]:
def run(*args):
    print('$', ' '.join(map(str, args)))
    return subprocess.run([str(x) for x in args], text=True, check=True).returncode

run('make', '-j2', 'all')
run('make', 'test')
run('make', 'test-parallel')
print('Build and correctness checks: PASS')


$ make -j2 all
make: Nothing to be done for `all'.
$ make test
./build/test_pagerank
[PASS] four-vertex graph converges
[PASS] four-vertex iteration count
[PASS] rank[0]
[PASS] rank[1]
[PASS] rank[2]
[PASS] isolated rank[3]
[PASS] rank sum is one
[PASS] four-vertex residual
[PASS] all-dangling graph converges
[PASS] all-dangling graph stops after one iteration
[PASS] all-dangling graph remains uniform
[PASS] duplicate edge is removed
[PASS] single self-loop has rank one
[PASS] declared dense isolated vertices are reconstructed
[PASS] isolated-vertex input preserves its edge
[PASS] reconstructed vertex IDs are dense and zero-based

Passed: 16
Failed: 0
$ make test-parallel
./build/pagerank --input tests/data/toy_graph.txt --mode serial --verify
mode                  = serial
dataset               = tests/data/toy_graph.txt
mpi ranks             = 1
threads per rank      = 1
partition             = vertices
score exchange        = full
vertices              = 4
edges                 = 4


mode                  = mpi
dataset               = tests/data/toy_graph.txt
mpi ranks             = 4
threads per rank      = 1
partition             = vertices
score exchange        = full
vertices              = 4
edges                 = 4
load seconds          = 3.8833e-05
distribution seconds  = 0.003548
dependency seconds    = 0
solve seconds         = 0.000168
compute seconds (max) = 5e-06
Allreduce seconds (max)= 0.000119
Allgatherv seconds (max)= 4.3e-05
score exchange seconds (max)= 4.3e-05
score pack seconds (max)= 0
score values / iteration= 12
score state bytes (max)= 72
update imbalance      = 1.6
work imbalance        = 1.5
iterations            = 53
converged             = yes
last L1 change        = 6.24583718078e-13
one-step residual     = 5.30964161527e-13
rank sum error        = 1.11022302463e-16
serial L1 error       = 0
validation            = PASS
top 4 ranks:
  1. original_id=2 rank=0.378475867453
  2. original_id=0 rank=0.369323534954
  3. original_id=1 rank=0.

mode                  = mpi
dataset               = tests/data/toy_graph.txt
mpi ranks             = 4
threads per rank      = 1
partition             = vertices
score exchange        = demand
vertices              = 4
edges                 = 4
load seconds          = 4.4667e-05
distribution seconds  = 0.004305
dependency seconds    = 0.0014
solve seconds         = 9.1e-05
compute seconds (max) = 7e-06
Allreduce seconds (max)= 7e-05
Allgatherv seconds (max)= 0
score exchange seconds (max)= 3.5e-05
score pack seconds (max)= 2e-06
score values / iteration= 4
score state bytes (max)= 124
update imbalance      = 1.33333333333
work imbalance        = 1.5
iterations            = 53
converged             = yes
last L1 change        = 6.24583718078e-13
one-step residual     = 5.30964161527e-13
rank sum error        = 2.22044604925e-16
serial L1 error       = 0
validation            = PASS
top 4 ranks:
  1. original_id=2 rank=0.378475867453
  2. original_id=0 rank=0.369323534954
  3. original_i

mode                  = hybrid
dataset               = tests/data/toy_graph.txt
mpi ranks             = 2
threads per rank      = 2
partition             = vertices
score exchange        = full
vertices              = 4
edges                 = 4
load seconds          = 6.9667e-05
distribution seconds  = 6.9e-05
dependency seconds    = 0
solve seconds         = 0.005588
compute seconds (max) = 0.002109
Allreduce seconds (max)= 0.003481
Allgatherv seconds (max)= 3.6e-05
score exchange seconds (max)= 3.6e-05
score pack seconds (max)= 0
score values / iteration= 4
score state bytes (max)= 80
update imbalance      = 1.01176470588
work imbalance        = 1
iterations            = 53
converged             = yes
last L1 change        = 6.24583718078e-13
one-step residual     = 5.30964161527e-13
rank sum error        = 1.11022302463e-16
serial L1 error       = 0
validation            = PASS
top 4 ranks:
  1. original_id=2 rank=0.378475867453
  2. original_id=0 rank=0.369323534954
  3. original_

mode                  = hybrid
dataset               = tests/data/toy_graph.txt
mpi ranks             = 2
threads per rank      = 2
partition             = in-edges
score exchange        = demand
vertices              = 4
edges                 = 4
load seconds          = 5.5708e-05
distribution seconds  = 0.00256
dependency seconds    = 0.001423
solve seconds         = 0.00166
compute seconds (max) = 0.001431
Allreduce seconds (max)= 0.000225
Allgatherv seconds (max)= 0
score exchange seconds (max)= 1.2e-05
score pack seconds (max)= 1e-06
score values / iteration= 3
score state bytes (max)= 108
update imbalance      = 1.00674157303
work imbalance        = 1
iterations            = 53
converged             = yes
last L1 change        = 6.24583718078e-13
one-step residual     = 5.30964161527e-13
rank sum error        = 2.22044604925e-16
serial L1 error       = 0
validation            = PASS
top 4 ranks:
  1. original_id=2 rank=0.378475867453
  2. original_id=0 rank=0.369323534954
  3. or

mode                  = mpi
dataset               = tests/data/isolated_header_graph.txt
mpi ranks             = 4
threads per rank      = 1
partition             = vertices
score exchange        = demand
vertices              = 4
edges                 = 1
load seconds          = 7.1375e-05
distribution seconds  = 0.00443
dependency seconds    = 0.001436
solve seconds         = 9.2e-05
compute seconds (max) = 2e-06
Allreduce seconds (max)= 7e-05
Allgatherv seconds (max)= 0
score exchange seconds (max)= 1.5e-05
score pack seconds (max)= 2e-06
score values / iteration= 1
score state bytes (max)= 96
update imbalance      = 4
work imbalance        = 1.6
iterations            = 19
converged             = yes
last L1 change        = 2.48745468667e-13
one-step residual     = 5.28188603965e-14
rank sum error        = 1.11022302463e-16
serial L1 error       = 1.94289029309e-16
validation            = PASS
top 4 ranks:
  1. original_id=1 rank=0.381443298969
  2. original_id=0 rank=0.20618556701


mode                  = mpi
dataset               = tests/data/isolated_header_graph.txt
mpi ranks             = 8
threads per rank      = 1
partition             = vertices
score exchange        = demand
vertices              = 4
edges                 = 1
load seconds          = 8.3e-05
distribution seconds  = 0.004153
dependency seconds    = 0.001503
solve seconds         = 8e-05
compute seconds (max) = 4e-06
Allreduce seconds (max)= 6.8e-05
Allgatherv seconds (max)= 0
score exchange seconds (max)= 2.5e-05
score pack seconds (max)= 3e-06
score values / iteration= 1
score state bytes (max)= 160
update imbalance      = 3.19999999999
work imbalance        = 3.2
iterations            = 19
converged             = yes
last L1 change        = 2.48745468667e-13
one-step residual     = 5.28188603965e-14
rank sum error        = 1.11022302463e-16
serial L1 error       = 1.94289029309e-16
validation            = PASS
top 0 ranks:
Build and correctness checks: PASS


## 3. Four Experimental Datasets

The experiments use four graphs: Google, BerkStan, English Wikipedia 2013, and UK-2002. Every run reads the same `PRCSR001` binary format so text parsing does not affect algorithm timing.

In [4]:
DATASET_ORDER = ['web-Google', 'web-BerkStan', 'enwiki-2013', 'uk-2002']
records = []
for name in DATASET_ORDER:
    data_path = ROOT / 'data/processed' / f'{name}.prcsr'
    meta_path = data_path.with_suffix('.prcsr.json')
    assert data_path.exists() and meta_path.exists(), name
    meta = json.loads(meta_path.read_text())
    records.append({
        'dataset': name,
        'vertices': int(meta['vertices']),
        'edges': int(meta.get('edges', meta.get('unique_edges'))),
        'CSR size (GiB)': data_path.stat().st_size / 1024**3,
        'format': meta['format'],
    })
datasets = pd.DataFrame(records)
dataset_table = datasets.copy()
dataset_table['vertices'] = dataset_table['vertices'].map('{:,}'.format)
dataset_table['edges'] = dataset_table['edges'].map('{:,}'.format)
dataset_table['CSR size (GiB)'] = dataset_table['CSR size (GiB)'].map('{:.3f}'.format)
display(dataset_table)
assert datasets['format'].eq('PRCSR001').all()


,dataset,vertices,edges,CSR size (GiB),format
0,web-Google,"875,713","5,105,039",0.035,PRCSR001
1,web-BerkStan,"685,230","7,600,595",0.041,PRCSR001
2,enwiki-2013,"4,203,323","101,311,614",0.456,PRCSR001
3,uk-2002,"18,520,486","298,113,762",1.456,PRCSR001


## 4. CSR File Integrity

Compare the SHA-256 digest of each dataset with `data/processed/SHA256SUMS`. Reading the large files may take a short time but does not modify them.

In [5]:
expected = {}
for line in (ROOT / 'data/processed/SHA256SUMS').read_text().splitlines():
    digest, filename = line.split()
    expected[filename] = digest

checks = []
for name in DATASET_ORDER:
    path = ROOT / 'data/processed' / f'{name}.prcsr'
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    actual = digest.hexdigest()
    checks.append({'file': path.name, 'SHA-256 match': actual == expected[path.name]})
checksum_table = pd.DataFrame(checks)
display(checksum_table)
assert checksum_table['SHA-256 match'].all()
print('Environment and data checks: PASS')


,file,SHA-256 match
0,web-Google.prcsr,True
1,web-BerkStan.prcsr,True
2,enwiki-2013.prcsr,True
3,uk-2002.prcsr,True


Environment and data checks: PASS


Next, run `01_local_experiments.ipynb` to inspect the three implementations on all four datasets on the Mac.